

# **🌍 City Travel Assistant with Agno Agent Tools**

---

## 📚 Notebook Overview

This notebook showcases how to build a **conversational travel assistant** using the `agno` framework. The agent uses **custom tool functions** to provide useful information for tourists about different cities worldwide. It supports tool-based reasoning for answering user queries by calling the appropriate tool function under the hood.

---

## ✅ What You'll Find in This Notebook

### 🔧 Tool Definitions
A set of custom tools are defined to serve travel-related queries, including:

- `get_city_info(city)` – Returns a tourist-friendly description of the given city
- `find_hotels(city)` – Lists two top hotel recommendations in the city
- `estimate_flight_cost(city)` – Provides an estimated round-trip flight cost in USD
- `get_weather(city)` – Shares a dummy 3-day weather forecast for the city

These tools simulate a basic travel recommendation system.

---

### 🤖 Agent Setup using `agno.agent.Agent`
- The notebook uses the `Agent` class from the `agno` Python library
- It automatically routes user queries to the correct tool function based on intent
- Designed to mimic function-calling behavior for travel-related assistance

---

### 🔁 User Interaction Flow
- Users can enter queries like:
  - "Tell me about Tokyo"
  - "Show hotels in Paris"
  - "What's the weather in London?"
- The agent interprets the query and selects the right tool function
- Results are returned in plain text as simulated responses

---
### 🧪 Response Evaluation with llumo
- Uses the llumo Python library to evaluate LLM-generated responses
---

## 🧠 Why Use This Notebook?

This notebook is ideal for:

- Learning how to implement tool-based agents using `agno`
- Building simple domain-specific assistants (like travel or tourism)
- Practicing function routing and modular tool design for LLM workflows
- Evaluating agent performance using standardized metrics

---


In [ ]:
!pip install agno -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.9/730.9 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.0/236.0 kB 8.9 MB/s eta 0:00:00


In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "Your own OpenAI Api"


## 🔧 **Setup: Defining the Agno Agent** 🧑‍💻

We first define the tools for our **Agno** agent that can provide basic travel-related information. These include:

- 🏙️ **City Info**
- 🏨 **Hotel Listings**
- ✈️ **Flight Estimates**
- 🌤️ **Weather Forecasts**

---


In [ ]:
import json
from agno.agent import Agent

# --- Tools ---

def get_city_info(city: str) -> str:
    """
    Gives basic info about a city for tourists.

    Args:
        city (str): Name of the city.

    Returns:
        str: Description of the city.
    """
    city_data = {
        "Paris": "Paris, the capital of France, is a top tourist destination known for the Eiffel Tower, the Louvre Museum, Notre-Dame Cathedral, Seine river cruises, and romantic café culture.",
        "Tokyo": "Tokyo offers a unique blend of tradition and modernity with attractions like the Meiji Shrine, Asakusa Temple, Akihabara electronics district, and Shibuya Crossing. Tourists also enjoy sushi, anime culture, and cherry blossoms in spring.",
        "New York": "New York City is famous for landmarks like Times Square, Central Park, the Statue of Liberty, the Empire State Building, and Broadway shows. It offers a fast-paced, multicultural experience for every kind of traveler.",
        "London": "London is a historic and cultural hub with sites like Buckingham Palace, Big Ben, the British Museum, the Tower of London, and the London Eye. Tourists enjoy double-decker buses, afternoon tea, and West End theatres.",
        "Dubai": "Dubai impresses tourists with the Burj Khalifa, Dubai Mall, Palm Jumeirah, desert safaris, and luxury resorts. It offers a blend of futuristic architecture and traditional Arabian experiences.",
        "Sydney": "Sydney is a coastal gem with the Sydney Opera House, Harbour Bridge, Bondi Beach, and Taronga Zoo. Visitors enjoy ferry rides, coastal walks, and the laid-back Australian lifestyle."
    }
    return city_data.get(city, "No info found for this city.")

def find_hotels(city: str) -> str:
    """
    Lists top 2 dummy hotel options in a city.

    Args:
        city (str): Name of the city.

    Returns:
        str: List of top hotels.
    """
    hotels = {
        "Paris": ["Hotel Le Meurice", "Hotel Lutetia"],
        "Tokyo": ["Park Hyatt Tokyo", "Shinjuku Granbell Hotel"],
        "New York": ["The Plaza", "CitizenM Times Square"],
        "London": ["The Savoy", "The Ritz London"],
        "Dubai": ["Burj Al Arab", "Atlantis The Palm"],
        "Sydney": ["Shangri-La Hotel", "Four Seasons Sydney"]
    }
    return f"Top hotels in {city}: {', '.join(hotels.get(city, ['No data']))}"

def estimate_flight_cost(city: str) -> str:
    """
    Returns a dummy round-trip flight cost in USD.

    Args:
        city (str): Name of the city.

    Returns:
        str: Estimated cost.
    """
    flights = {
        "Paris": "$800",
        "Tokyo": "$1000",
        "New York": "$500",
        "London": "$750",
        "Dubai": "$900",
        "Sydney": "$1200"
    }
    return f"Estimated round-trip flight to {city} costs {flights.get(city, 'N/A')}"

def get_weather(city: str) -> str:
    """
    Returns dummy weather for the next 3 days.

    Args:
        city (str): Name of the city.

    Returns:
        str: Weather forecast.
    """
    weather = {
        "Paris": "Day 1: Sunny, 22°C\nDay 2: Rainy, 18°C\nDay 3: Cloudy, 20°C",
        "Tokyo": "Day 1: Cloudy, 25°C\nDay 2: Rainy, 23°C\nDay 3: Sunny, 26°C",
        "New York": "Day 1: Sunny, 27°C\nDay 2: Thunderstorm, 22°C\nDay 3: Sunny, 24°C",
        "London": "Day 1: Rainy, 15°C\nDay 2: Cloudy, 16°C\nDay 3: Drizzle, 14°C",
        "Dubai": "Day 1: Sunny, 35°C\nDay 2: Sunny, 36°C\nDay 3: Windy, 33°C",
        "Sydney": "Day 1: Partly Cloudy, 21°C\nDay 2: Sunny, 23°C\nDay 3: Rainy, 20°C"
    }
    return weather.get(city, "No weather data available.")



In [ ]:
tool_descriptions = {
    "get_city_info": "Gives basic info about a city for tourists.",
    "find_hotels": "Lists top 2 dummy hotel options in a city.",
    "estimate_flight_cost": "Returns a dummy round-trip flight cost in USD.",
    "get_weather": "Returns dummy weather for the next 3 days."
}


## 🤖 Create the City Travel Agent Team - Agno

Set up the `Agent` with all the travel-related tools. This agent will handle user queries by calling the appropriate tool based on intent.


In [ ]:
agent_team = Agent(
    tools=[get_city_info, find_hotels, estimate_flight_cost, get_weather],
    show_tool_calls=True,
    markdown=True,
    add_history_to_messages=False,
)

## **✂️ Message History: Chunk and Format Conversation History**
This function processes agent conversation history by removing system messages and formatting the rest into a structured format suitable for evaluation or logging.


In [ ]:
def chunk_conversation(messages):
    chunks = []

    for message in messages:
        if message.role == "system":
            continue  # Ignore system messages

        elif message.role == "tool":
             message_obj = {
            "role": "tool",
            "tool_name": message.tool_name.replace("transfer_task_to_",""),
            "parts": {"text": message.content},
        }

        else:
          message_obj = {
              "role": message.role,
              "parts": {"text": message.content}
          }
        chunks.append(message_obj)
    return chunks



## 🛠️ **Running the Agent** 🏃‍♂️

Let’s process the queries through the **Agno** agent to get responses. The results will help us evaluate the agent's performance.


In [ ]:
# --- Example Query ---
import time

sample_queries = [
    "Tell me about tourist attractions in Tokyo.",
    "How much does it cost to fly to Dubai?",
    "Can you book a flight for India for tomorrow?",

]

history = []
for question in sample_queries:
  time.sleep(4)
  # agent_team.print_response(question, stream = False)
  res = agent_team.run(question)
  history.append(chunk_conversation(res.messages))


## 💾 **Storing the Responses** 📊

The agent’s responses, along with the tools used, will be stored in a CSV file for easy access and future analysis.


In [ ]:
import pandas as pd

df = pd.DataFrame()
df["query"] = sample_queries
df["output"] = [i[-1]["parts"]["text"] for i in history]
df["messageHistory"] = history
df["tools"] = f"{tool_descriptions}"

In [ ]:
df

,query,output,messageHistory,tools
0,Tell me about tourist attractions in Tokyo.,Tokyo offers a unique blend of tradition and m...,"[{'role': 'user', 'parts': {'text': 'Tell me a...",{'get_city_info': 'Gives basic info about a ci...
1,How much does it cost to fly to Dubai?,The estimated round-trip flight cost to Dubai ...,"[{'role': 'user', 'parts': {'text': 'How much ...",{'get_city_info': 'Gives basic info about a ci...
2,Can you book a flight for India for tomorrow?,"I can't book flights, but I can help estimate ...","[{'role': 'user', 'parts': {'text': 'Can you b...",{'get_city_info': 'Gives basic info about a ci...


In [ ]:
df.to_csv("agno_data.csv")

##**🧠 Evaluate Agent Responses using LlumoClient**
- Uses the llumo Python library to evaluate LLM-generated responses
- LlumoClient is used to score responses on criteria such as:
  - Tool usage correctness
  - Overall quality, completeness and correctness
- Helps analyze and benchmark the performance of the conversational agent

### **STEP 1: ⬇ Import necessary modules**

In [ ]:
!pip install llumo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.78.1
    Uninstalling openai-1.78.1:
      Successfully uninstalled openai-1.78.1


###**STEP 2: Initialize Llumo Client And Evaluate**

Import the data for evaluation

In [ ]:
import pandas as pd

# Importing the data frame that we had previousy created
df = pd.read_csv("agno_data.csv")

Converting the dataframe into a json format: [{},{}]


In [ ]:
data = df.to_dict(orient="records")

In [ ]:
from llumo import LlumoClient
from llumo.functionCalling import LlumoAgent

# 🔑 Initialize the LlumoClient with your LLUMO API key
client = LlumoClient(api_key="Enter your llumo api key")  # Replace with your Llumo Key



### **🛠️ Tool-Based Metrics**
- 🔧 Tool Reliability  
- 🪜 Stepwise Progression  
- 🎯 Tool Selection Accuracy  
- ✅ Final Task Alignment


In [3]:
# ✅ Use the client to evaluate agent responses using the collected DataFrame
results = client.evaluateAgentResponses(
    data, # data dict
    evals=["Tool Reliability","Stepwise Progression","Tool Selection Accuracy","Final Task Alignment"], # list of metrics
    outputColName = "output" # output column name. Default is "output"
    )

## 🏁 **Results Analysis** 📊

Once the responses are evaluated, we can interpret the results to understand the strengths and weaknesses of the agent's answers.


In [ ]:
results

,query,output,messageHistory,tools,Tool Reliability,Tool Reliability Reason,Stepwise Progression,Stepwise Progression Reason,Tool Selection Accuracy,Tool Selection Accuracy Reason,Final Task Alignment,Final Task Alignment Reason
0,Tell me about tourist attractions in Tokyo.,Tokyo offers a unique blend of tradition and m...,"[{'role': 'user', 'parts': {'text': 'Tell me a...",{'get_city_info': 'Gives basic info about a ci...,99,The 'get_city_info' tool successfully executed...,100,The tool 'get_city_info' is relevant to the us...,99,"The assistant used only 'get_city_info', which...",99,The assistant's final response directly answer...
1,How much does it cost to fly to Dubai?,The estimated round-trip flight cost to Dubai ...,"[{'role': 'user', 'parts': {'text': 'How much ...",{'get_city_info': 'Gives basic info about a ci...,100,The `estimate_flight_cost` tool successfully r...,100,The tool 'estimate_flight_cost' is directly re...,100,The user asked about flight cost to Dubai. Th...,99,The assistant accurately provided the requeste...
2,Can you book a flight for India for tomorrow?,"I can't book flights, but I can help estimate ...","[{'role': 'user', 'parts': {'text': 'Can you b...",{'get_city_info': 'Gives basic info about a ci...,2,No tools were used. The assistant indicated it...,2,No tools were used to address the user's reque...,1,The user requested flight booking. The assist...,2,The assistant failed to book a flight as reque...


## 🎯 **Objective** 🧠

The primary goal of this notebook is to demonstrate how users can evaluate **Agno** agent responses using the **Llumo** platform. Through this process, users can gather insights and improve their agents based on performance metrics.

---

## 🚀 **Let’s Get Started** 👩‍💻

Follow the steps Above to explore how we use **Agno** for agent creation and **Llumo** for evaluating the responses. Let’s dive in!